# E3 · 01 — Datos comunes para los modelos

Prepara, una sola vez, todo lo que usan los cuatro modelos (SVM, XGBoost, CNN y BiLSTM), para que todos trabajen exactamente con los mismos textos, etiquetas y representaciones. Ningún notebook de modelos vuelve a calcular nada de esto: solo carga los ficheros de `data/e3/`.

| Fichero | Contenido |
|---|---|
| `datos_E3.jsonl` | Una fila por texto: `id`, `corpus`, `split`, `texto`, `tokens`, `raices_bow`, `etiqueta_deteccion`, `etiqueta_tipo`. **El orden de las filas es el de todas las matrices** |
| `clases_tipo.json` | Las clases del tipo, en orden (la posición es el número de la clase) |
| `tfidf.npz`, `vocabulario_tfidf.json` | TF-IDF de `raices_bow` |
| `features_extra.npy` | Vector medio de eswiki (300 columnas) + nº de `?` y nº de `!` |
| `vocabulario_secuencias.json`, `secuencias.npy`, `matriz_eswiki.npy` | Para las redes: palabra → índice, cada texto como índices y los vectores de eswiki de cada palabra |

El notebook se ejecuta desde la carpeta `E3/`.

## 0. Preparación

Se usan los cuatro corpus anotados ya preprocesados (notebook `E2/03`). Los sin falacia de Political (E3) cuentan como corpus `political`, porque salen de los mismos debates. El Congreso no se usa: no está anotado.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse

CARPETA_DATOS = Path("../data/processed")
CARPETA_SALIDA = Path("../data/e3")
CARPETA_SALIDA.mkdir(exist_ok=True)

CORPUS = {  # fichero → nombre del corpus en la tabla común
    "fallacyes_proto": "proto",
    "fallacyes_espont": "espont",
    "fallacyes_political": "political",
    "fallacyes_political_neg": "political",
}

## 1. Tabla común: una fila por texto

Los modelos clasifican el texto entero, así que los `tokens` y las `raices_bow` de sus oraciones se juntan en una sola lista.

In [2]:
def juntar(oraciones, campo):
    """Junta en una sola lista los tokens (o las raíces) de todas las oraciones del texto."""
    return [elemento for oracion in oraciones for elemento in oracion[campo]]


tablas = []
for fichero, corpus in CORPUS.items():
    tabla = pd.read_json(CARPETA_DATOS / f"{fichero}_procesado.jsonl", lines=True, dtype=False)
    tablas.append(pd.DataFrame({
        "id": tabla["id"],
        "corpus": corpus,
        "split": tabla["split"],
        "texto": tabla["texto"],
        "tokens": tabla["oraciones"].map(lambda oraciones: juntar(oraciones, "tokens")),
        "raices_bow": tabla["oraciones"].map(lambda oraciones: juntar(oraciones, "raices_bow")),
        "n_oraciones": tabla["oraciones"].str.len(),
        "etiquetas": tabla["etiquetas"],
    }))

datos = pd.concat(tablas, ignore_index=True)
es_train = datos["split"] == "train"
datos.groupby(["corpus", "split"]).size().unstack()

split,dev,test,train
corpus,,,
espont,273,266,1301
political,401,413,1870
proto,489,489,2341


## 2. Etiquetas de las dos tareas

- **Detección** (`etiqueta_deteccion`): `1` si el texto tiene alguna falacia y `0` si no. Entran todos los textos.
- **Tipo** (`etiqueta_tipo`): la lista de falacias, o `null` si el texto no entra en esta tarea:
  - Los textos sin falacia.
  - `otra_falacia` de las prototípicas: allí es razonamiento circular o intencional, mientras que en el test de Political es sobre todo pregunta compleja; la clase no significaría lo mismo en train y en test. Siguen en la detección.
  - Los 2 textos de dev con varias etiquetas: al evaluar, una de las dos siempre contaría como fallo. En test no hay ninguno.

Los 9 textos de train con varias etiquetas se quedan con su lista: cada notebook de modelos los duplica (una fila por etiqueta) con `explode`.

In [3]:
def etiqueta_tipo(corpus, split, etiquetas):
    """Lista de falacias para la tarea del tipo, o None si el texto no entra en ella."""
    if not etiquetas:
        return None
    if corpus == "proto" and "otra_falacia" in etiquetas:
        return None
    if split == "dev" and len(etiquetas) > 1:
        return None
    return etiquetas


datos["etiqueta_deteccion"] = (datos["etiquetas"].str.len() > 0).astype(int)
datos["etiqueta_tipo"] = [
    etiqueta_tipo(corpus, split, etiquetas)
    for corpus, split, etiquetas in zip(datos["corpus"], datos["split"], datos["etiquetas"])
]
CLASES_TIPO = sorted({clase for lista in datos["etiqueta_tipo"].dropna() for clase in lista})
print(len(CLASES_TIPO), "clases:", CLASES_TIPO)

pd.crosstab([datos["corpus"], datos["split"]], datos["etiqueta_deteccion"]).rename(columns={0: "sin falacia", 1: "falacia"})

11 clases: ['ad_hominem', 'ad_populum', 'apelacion_emocion', 'deduccion_invalida', 'distraccion', 'falacia_credibilidad', 'falsa_causalidad', 'falso_dilema', 'generalizacion_apresurada', 'hombre_de_paja', 'otra_falacia']


etiqueta_deteccion  sin falacia  falacia
corpus    split                         
espont    dev               132      141
          test              125      141
          train             660      641
political dev               110      291
          test               69      344
          train             554     1316
proto     dev               183      306
          test              183      306
          train             916     1425

In [4]:
# Textos de la tarea del tipo
tipo = datos[datos["etiqueta_tipo"].notna()]
pd.crosstab(tipo["corpus"], tipo["split"], margins=True)

split,dev,test,train,All
corpus,,,,
espont,141,141,641,923
political,289,344,1316,1949
proto,265,265,1234,1764
All,695,750,3191,4636


## 3. TF-IDF (para el SVM y el XGBoost)

Sobre las `raices_bow`, que ya vienen preparadas del notebook `E2/03` (minúsculas, sin stopwords salvo las que indican falacias, raíces, cifras como `<num>`). Se ajusta **solo con train** y con `min_df=2`: una raíz que solo aparece en un texto de train no ayuda a generalizar. Un único TF-IDF para las dos tareas.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(analyzer=lambda raices: raices, min_df=2)
tfidf.fit(datos.loc[es_train, "raices_bow"])
matriz_tfidf = tfidf.transform(datos["raices_bow"])
print("TF-IDF:", matriz_tfidf.shape[0], "textos x", matriz_tfidf.shape[1], "raíces")

TF-IDF: 7843 textos x 4503 raíces


## 4. Vectores de eswiki

Word2Vec preentrenado con la Wikipedia en español (Wikipedia2Vec, 300 dimensiones), el mismo que en `E2/04`. La primera vez se descarga (4 GB) y se lee el fichero de texto, que tarda varios minutos; después se guarda en el formato de gensim (`.kv`), que se carga en segundos. Va a `models/`, que no se sube a GitHub: Álvaro no lo necesita, porque todo lo que sale de eswiki se guarda en `data/e3/`.

Las palabras de eswiki están en minúsculas, así que los tokens se pasan a minúsculas para buscarlas. Así, además, la mayúscula inicial no puede delatar a los sin falacia nuevos de Political (el 23 % de las falacias de Political empiezan a mitad de frase).

In [6]:
from gensim.models import KeyedVectors
from huggingface_hub import hf_hub_download

RUTA_KV = Path("../models/eswiki_300d.kv")

if RUTA_KV.exists():
    eswiki = KeyedVectors.load(str(RUTA_KV))
else:
    ruta_txt = hf_hub_download(repo_id="Word2vec/wikipedia2vec_eswiki_20180420_300d",
                               filename="eswiki_20180420_300d.txt")
    eswiki = KeyedVectors.load_word2vec_format(ruta_txt)
    RUTA_KV.parent.mkdir(exist_ok=True)
    eswiki.save(str(RUTA_KV))

minusculas = datos["tokens"].map(lambda tokens: [token.lower() for token in tokens])
print(len(eswiki), "palabras en eswiki")

C:\Users\asier.gomez\Proyectos\Clase\detector_falacias_contradicciones\nlp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1828809 palabras en eswiki


## 5. Features extra (para combinarlas con el TF-IDF)

Por texto, en este orden de columnas:
- 300 columnas: **media de los vectores de eswiki** de sus palabras (ceros si ninguna tiene vector).
- 2 columnas: **nº de `?` y nº de `!`**. En la E2 se midió que son mucho más frecuentes en las falacias: es retórica real.

La longitud (nº de tokens u oraciones) **no** se usa como feature: en nuestros datos indica de qué corpus viene el texto y, en Political, cómo cortamos los sin falacia (sección 7), no si hay falacia. Solo se calcula para la prueba de la sección 7.

In [7]:
def vector_medio(tokens):
    vectores = [eswiki[token] for token in tokens if token in eswiki.key_to_index]
    if not vectores:
        return np.zeros(eswiki.vector_size, dtype=np.float32)
    return np.mean(vectores, axis=0)


rasgos_forma = pd.DataFrame({
    "n_tokens": datos["tokens"].str.len(),
    "n_oraciones": datos["n_oraciones"],
    "n_interrogaciones": datos["texto"].str.count(r"\?"),
    "n_exclamaciones": datos["texto"].str.count("!"),
})
signos = rasgos_forma[["n_interrogaciones", "n_exclamaciones"]].to_numpy()
features_extra = np.hstack([np.stack(minusculas.map(vector_medio)), signos]).astype(np.float32)
print("Features extra:", features_extra.shape)

Features extra: (7843, 302)


## 6. Secuencias para las redes (CNN y BiLSTM)

- **Vocabulario:** las palabras de train y, además, las de dev y test **que tienen vector en eswiki**. No usa ninguna etiqueta: equivale a que el modelo conozca el vocabulario de eswiki, y aprovecha que una palabra no vista en train tiene un vector parecido al de otras que sí. `0` = relleno, `1` = palabra desconocida.
- **`MAX_LEN = 150` tokens:** los textos más largos se recortan y los más cortos se rellenan con `0`. Con 150 se recorta el 0,8 % de los textos (el 3 % de las falacias de Political); con 120 era el 5 % de esas falacias, y la falacia podría estar justo en la parte cortada.
- **Matriz de embeddings:** fila *i* = vector de eswiki de la palabra *i*. Las palabras sin vector empiezan con valores aleatorios pequeños (con la misma desviación que los de eswiki); el relleno, con ceros.

In [8]:
MAX_LEN = 150
RELLENO, DESCONOCIDA = 0, 1

palabras_train = {token for tokens in minusculas[es_train] for token in tokens}
palabras_resto_con_vector = {token for tokens in minusculas[~es_train] for token in tokens if token in eswiki.key_to_index}

vocabulario = {"<relleno>": RELLENO, "<desconocida>": DESCONOCIDA}
for palabra in sorted(palabras_train | palabras_resto_con_vector):
    vocabulario[palabra] = len(vocabulario)


def a_indices(tokens):
    indices = [vocabulario.get(token, DESCONOCIDA) for token in tokens][:MAX_LEN]
    return indices + [RELLENO] * (MAX_LEN - len(indices))


secuencias = np.array(minusculas.map(a_indices).tolist(), dtype=np.int32)
print("Vocabulario:", len(vocabulario), "palabras | secuencias:", secuencias.shape)

Vocabulario: 18437 palabras | secuencias: (7843, 150)


In [9]:
generador = np.random.default_rng(42)
matriz_eswiki = generador.normal(0, eswiki.vectors.std(), size=(len(vocabulario), eswiki.vector_size)).astype(np.float32)
matriz_eswiki[RELLENO] = 0

con_vector = [palabra for palabra in vocabulario if palabra in eswiki.key_to_index]
for palabra in con_vector:
    matriz_eswiki[vocabulario[palabra]] = eswiki[palabra]

print(f"Palabras del vocabulario con vector de eswiki: {len(con_vector)} de {len(vocabulario)}")
tokens_train = [token for tokens in minusculas[es_train] for token in tokens]
print(f"Tokens de train con vector de eswiki: {np.mean([token in eswiki.key_to_index for token in tokens_train]):.1%}")

Palabras del vocabulario con vector de eswiki: 17560 de 18437
Tokens de train con vector de eswiki: 88.2%


## 7. Prueba del atajo

Los sin falacia de Political los construimos nosotros (notebook `E2/01`, secciones 11 y 12). Si se distinguieran de las falacias por la **forma** y no por el contenido, los modelos podrían aprender esa diferencia en vez de las falacias.

Se entrena un árbol de decisión pequeño **solo con los 4 rasgos de forma** para separar falacia / sin falacia en Political, y se compara en dev con la referencia de decir siempre "falacia". Si apenas la supera, la forma no delata la clase. Si la supera bastante, se mira qué rasgo lo hace: la longitud sería un artefacto de nuestro corte; `?` y `!` son retórica real de las falacias.

In [10]:
from sklearn.metrics import f1_score
from sklearn.tree import DecisionTreeClassifier

political = datos["corpus"] == "political"
en_train, en_dev = political & es_train, political & (datos["split"] == "dev")

arbol = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=0)
arbol.fit(rasgos_forma[en_train], datos.loc[en_train, "etiqueta_deteccion"])

y_dev = datos.loc[en_dev, "etiqueta_deteccion"]
print(f"F1 macro en dev solo con la forma: {f1_score(y_dev, arbol.predict(rasgos_forma[en_dev]), average='macro'):.2f}")
print(f"F1 macro en dev diciendo siempre 'falacia': {f1_score(y_dev, np.ones(len(y_dev)), average='macro'):.2f}")
print("Peso de cada rasgo en el árbol:", dict(zip(rasgos_forma.columns, arbol.feature_importances_.round(2))))

F1 macro en dev solo con la forma: 0.53
F1 macro en dev diciendo siempre 'falacia': 0.42
Peso de cada rasgo en el árbol: {'n_tokens': np.float64(0.9), 'n_oraciones': np.float64(0.09), 'n_interrogaciones': np.float64(0.02), 'n_exclamaciones': np.float64(0.0)}


## 8. Guardar

In [11]:
COLUMNAS = ["id", "corpus", "split", "texto", "tokens", "raices_bow", "etiqueta_deteccion", "etiqueta_tipo"]
with open(CARPETA_SALIDA / "datos_E3.jsonl", "w", encoding="utf-8") as fichero:
    for registro in datos[COLUMNAS].astype(object).to_dict(orient="records"):
        fichero.write(json.dumps(registro, ensure_ascii=False) + "\n")


def guardar_json(objeto, nombre):
    with open(CARPETA_SALIDA / nombre, "w", encoding="utf-8") as fichero:
        json.dump(objeto, fichero, ensure_ascii=False)


guardar_json(CLASES_TIPO, "clases_tipo.json")
guardar_json(tfidf.get_feature_names_out().tolist(), "vocabulario_tfidf.json")
guardar_json(vocabulario, "vocabulario_secuencias.json")
sparse.save_npz(CARPETA_SALIDA / "tfidf.npz", matriz_tfidf)
np.save(CARPETA_SALIDA / "features_extra.npy", features_extra)
np.save(CARPETA_SALIDA / "secuencias.npy", secuencias)
np.save(CARPETA_SALIDA / "matriz_eswiki.npy", matriz_eswiki)

for ruta in sorted(CARPETA_SALIDA.iterdir()):
    print(f"{ruta.name}: {ruta.stat().st_size / 1e6:.1f} MB")

clases_tipo.json: 0.0 MB
datos_E3.jsonl: 6.1 MB
features_extra.npy: 9.5 MB
matriz_eswiki.npy: 22.1 MB
secuencias.npy: 4.7 MB
tfidf.npz: 1.0 MB
vocabulario_secuencias.json: 0.3 MB
vocabulario_tfidf.json: 0.0 MB
